# **Variational Inference: Approximating a Gaussian Posterior Using ELBO**

## **Problem Statement**
We aim to approximate an intractable posterior using **Variational Inference (VI)** by maximizing the **Evidence Lower Bound (ELBO)**.

### **Step 1: Define the True Distribution (`ptrue`)**
In this example, the **true distribution (`ptrue`)** also serves as the **prior distribution**:

$$
p(z) = 0.5 \cdot \mathcal{N}(1,1) + 0.5 \cdot \mathcal{N}(10,1.5)
$$

### **Step 2: Compute the Log-Likelihood**
The likelihood function **models how $x$ is generated given $z$**:

$$
p(x | z) = \mathcal{N}(x | z, \sigma^2)
$$

Implement `log_likelihood(x, z)`.

### **Step 3: Compute KL Divergence via Monte Carlo Sampling**
Since an **analytical form** of $D_{KL}(q || p)$ is often intractable, we approximate it using **Monte Carlo estimation**:

$$
D_{KL}(q || p) \approx \frac{1}{N} \sum_{i=1}^{N} \left( \log q(z_i) - \log p(z_i) \right), \quad z_i \sim q(z)
$$

Implement `kl_sampling_inverse(params, p, samples)`.

### **Step 4: Compute ELBO**
ELBO is given by:

$$
\mathcal{L} = \mathbb{E}_{q(z)} [\log p(x | z)] - D_{KL}(q(z) || p(z))
$$

Implement `elbo(params, x_batch, samples)`.

### **Step 5: Perform Optimization (`fit`)**
Train a **variational distribution** $q(z) = \mathcal{N}(\mu, \sigma^2)$ using **gradient-based optimization**.

### **Step 6: Visualization**
- **ELBO Convergence Plot**: Check if ELBO increases and stabilizes.
- **Posterior Approximation**: Compare learned $q(z)$ with `ptrue`.

---


In [ ]:
import jax.numpy as jnp
import jax
try:
    import distrax #defining probabilistic models (Gaussian)
except:
    %pip install -qq distrax
    import distrax
import optax
import matplotlib.pyplot as plt
import seaborn as sns

**Bayesian inference**
$$
p(z | x) = \frac{p(x | z) p(z)}{p(x)}
$$


Posterior Distribution: $p(z | x)$  
Likelihood: $p(x | z)$  
Prior Distribution: $p(z)$  
Evidence (Marginal Likelihood): $p(x)$  

In [ ]:
# true distribution (also used as the prior)
ptrue = distrax.MixtureSameFamily(
    mixture_distribution=distrax.Categorical(probs=[0.5, 0.5]),
    components_distribution=distrax.Normal(loc=[1, 10], scale=[1, 1.5]),
)

In [ ]:
# TODO: likelihood function p(x | z) ~ N(z, 0.5)
def log_likelihood(x, z):

In [ ]:
# TODO: Approximate KL divergence D_KL(q || p) using Monte Carlo sampling.
def kl_sampling_inverse(params, p, samples=100000):

In [ ]:
# TODO: Compute the ELBO for Variational Inference.
def elbo(params, x_batch, samples=1000):
  # Use kl_sampling_inverse

In [ ]:
# TODO: Optimize Variational Distribution (Fit q(z))
def fit_vi(params, optimizer, n_itr, x_batch):
  return opt_params, loss_history

In [ ]:
### Train and Visualize Results

# Define multiple observed values
x_obs = jnp.array([1.5, 2.0, 3.0, 4.5])  # Sampled observations

# Initialize variational parameters (mu, log_sigma)
params = jnp.array([5.0, 1.0])  # Reasonable starting values

# Define optimizer
optimizer = optax.adam(learning_rate=0.01)
n_itr = 400  # Number of iterations

# Train the variational model
opt_params, loss_history = fit_vi(params, optimizer, n_itr, x_obs)
# Extract optimized variational parameters
mu_opt, log_sigma_opt = opt_params
sigma_opt = jnp.exp(log_sigma_opt)

# ELBO Convergence Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Loss curve
ax1.plot(loss_history)
ax1.set_title("ELBO Convergence")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Negative ELBO")

# Posterior Approximation Plot
z_vals = jnp.linspace(-3, 15, 1000)
q_approx = distrax.Normal(mu_opt, sigma_opt).prob(z_vals)
prior_vals = ptrue.prob(z_vals)

ax2.plot(z_vals, prior_vals, label="Prior p(z)", linestyle="--")
ax2.plot(z_vals, q_approx, label="Variational q(z)", color="green")
ax2.set_title("Variational Approximation of Posterior")
ax2.set_xlabel("z")
ax2.set_ylabel("Density")
ax2.legend()

sns.despine()
plt.show()

# Print optimized parameters
print(f"Optimized Variational Mean (mu): {mu_opt:.3f}")
print(f"Optimized Variational Std (sigma): {sigma_opt:.3f}")